# 1. 環境準備與模組載入

載入量化交易模型所需的數據處理、機器學習（LightGBM）、評估指標以及繪圖視覺化套件。


In [131]:
import os
import numpy as np
import pandas as pd
import pandas_ta as ta
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# 2. 實驗目標個股定義

定義納入多股聯合訓練與回測的台灣股市標的（以 0050 成分股為主）。


In [132]:
# 核心實驗個股：0050
stock_ids = ['1216', '1303', '2059', '2301', '2303', '2308', '2317', '2327', '2330', '2344',
       '2345', '2357', '2360', '2368', '2382', '2383', '2395', '2408', '2412', '2449',
       '2454', '2603', '2880', '2881', '2882', '2883', '2884', '2885', '2886', '2887',
       '2890', '2891', '2892', '3008', '3017', '3037', '3045', '3231', '3443', '3653',
       '3661', '3665', '3711', '4904', '4958', '5880', '6505', '6669', '7769', '8046']
# 備用擴展個股池（目前註解排除）
stock_ids += [
       '1101', '1102', '1301', '1326', '1402', '1476', '1503', '1504', '1513', '1519',
       '1560', '1590', '1605', '1717', '2105', '2207', '2312', '2313', '2324', '2337',
       '2347', '2353', '2354', '2356', '2371', '2376', '2377', '2379', '2385', '2404',
       '2409', '2455', '2474', '2492', '2542', '2609', '2610', '2615', '2618', '2633',
       '2645', '2801', '2812', '2834', '2912', '3005', '3023', '3034', '3036', '3044'
       ]

# 3. 交易策略與成本參數配置

設定量化策略的核心控管參數，包含停利點、信心門檻，以及貼近台股真實市場的摩擦成本（手續費折讓與證交稅）。


In [133]:
TAKE_PROFIT = 0.09         # 動態停利點：相對於買入價上漲 9%
STOP_LOSS = -0.15          # 動態停損點：相對於買入價下跌 15%
BUY_THRESHOLD = 0.70      # 模型發出買入訊號的最低信心度門檻 (70%)

# 台股交易摩擦成本計算
TAIWAN_FEE_RATE = 0.001425  # 券商法定基本手續費率
TAX_RATE = 0.003            # 證券交易稅率 (0.3%)
FEE_DISCOUNT = 0.6          # 券商手續費折讓（6折）

TOTAL_COST_RATIO = (TAIWAN_FEE_RATE * FEE_DISCOUNT * 2) + TAX_RATE
ENTRY_FEE = TAIWAN_FEE_RATE * FEE_DISCOUNT            # 買進手續費成本
EXIT_FEE = (TAIWAN_FEE_RATE * FEE_DISCOUNT) + TAX_RATE # 賣出綜合交易成本（含稅）

INITIAL_CAPITAL = 200000    # 回測初始全額本金

# 4. 資料管線、特徵工程與多股聯合模型訓練

1. 讀取真實歷史 CSV，建立未來 10 天滾動最大報酬作為 Y Label。
2. 進行前 30 天歷史日 K 資料平坦化（Lagging），建立高維度特徵矩陣 X。
3. 實施嚴格的橫截面時間序列切分（Time-Series Split），防範 Look-ahead Bias。
4. 使用 LightGBM 進行聯合訓練，並對測試集輸出預測信心度。


In [142]:
# ====================== 先定義技術指標函數 ======================
def add_technical_features(df):
    """計算技術指標 - 必須在分組後執行"""
    df = df.copy()
    
    # 判斷分組欄位（相容 stock_id 或 ticker）
    group_col = 'stock_id' if 'stock_id' in df.columns else ('ticker' if 'ticker' in df.columns else None)
    
    processed_groups = []
    
    if group_col:
        for name, group in df.groupby(group_col):
            g = group.copy()
            g = g.sort_values('date').reset_index(drop=True)
            
            # 計算技術指標
            g['rsi_14'] = ta.rsi(g['close'], length=14)
            
            # MACD
            macd_df = ta.macd(g['close'])
            if macd_df is not None:
                g['macd'] = macd_df['MACD_12_26_9']
                g['macd_signal'] = macd_df['MACDs_12_26_9']
            
            # Bollinger Bands 
            bb = ta.bbands(g['close'], length=20, std=2)
            if bb is not None:
                g['bb_lower'] = bb['BBL_20_2.0_2.0']   
                g['bb_middle'] = bb['BBM_20_2.0_2.0']
                g['bb_upper'] = bb['BBU_20_2.0_2.0']
                g['bb_width'] = g['bb_upper'] - g['bb_lower']
                g['bb_percent'] = bb['BBP_20_2.0_2.0']
                
            g['atr_14'] = ta.atr(g['max'], g['min'], g['close'], length=14)
            
            # 動量 / 趨勢 / 波動
            g['return_1'] = g['close'].pct_change()
            g['mom_5'] = g['close'].pct_change(5)
            g['mom_10'] = g['close'].pct_change(10)
            g['ma_20'] = g['close'].rolling(20).mean()
            g['price_vs_ma20'] = g['close'] / g['ma_20'] - 1
            g['vol_ma20'] = g['Trading_Volume'].rolling(20).mean()
            g['vol_ratio_20'] = g['Trading_Volume'] / g['vol_ma20']
            g['vol_20'] = g['return_1'].rolling(20).std()
            
            processed_groups.append(g)
            
        # 重新合併所有分組，此時新欄位會被完整保留
        df = pd.concat(processed_groups, ignore_index=True)
        
    else:
        # 若無分組欄位，直接當作單一股票計算
        df = df.sort_values('date').reset_index(drop=True)
        # [將上方指標計算複製至此...]
        
    return df


# ====================== 修改後的 load_all_stocks_with_lag ======================
def load_all_stocks_with_lag(stock_list: list, lag_days: int = 20):   # ← 建議改成 20
    all_stock_dfs = []
    for ticker in stock_list:
        file_path = f'../data/{ticker}_stock_data.csv'
        if not os.path.exists(file_path):
            print(f"警告: 找不到 {ticker} 的資料，跳過。")
            continue
            
        df = pd.read_csv(file_path)
        
        # 加入技術指標
        # df = add_technical_features(df)
        df['ticker'] = ticker
        
        # Label
        df['future_max_high'] = df['max'].shift(-10).rolling(window=10, min_periods=1).max()
        df['future_max_return'] = (df['future_max_high'] - df['close']) / df['close']
        df['label'] = (df['future_max_return'] >= TAKE_PROFIT).astype(int)
        
        # 特徵平坦化
        exclude_cols = ['date', 'label', 'future_max_high', 'future_max_return', 'ticker', 'stock_id']
        base_features = [col for col in df.columns if col not in exclude_cols]
        feature_dict = {}
        for lag in range(lag_days):
            lagged = df[base_features].shift(lag)
            for col in base_features:
                if col in ['open', 'max', 'min', 'close']:
                    feature_dict[f"{col}_lag_{lag}"] = (lagged[col] - df['close']) / df['close']
                elif col == 'Trading_Volume':
                    feature_dict[f"volume_lag_{lag}"] = lagged[col] / (df['Trading_Volume'] + 1e-8)
                else:
                    # 技術指標直接 lag
                    feature_dict[f"{col}_lag_{lag}"] = lagged[col]
        
        X_all = pd.DataFrame(feature_dict)
        
        df_final = pd.concat([
            df[['date', 'label', 'future_max_return', 'ticker']], 
            X_all
        ], axis=1)
        
        df_final = df_final.dropna().reset_index(drop=True)
        df_final['ticker'] = df_final['ticker'].astype('category')
        
        all_stock_dfs.append(df_final)
        
    full_dataset = pd.concat(all_stock_dfs, axis=0, ignore_index=True)
    return full_dataset


# ====================== 使用方式 ======================
print("開始串接所有股票資料...")
total_df = load_all_stocks_with_lag(stock_list=stock_ids, lag_days=20)  # ← 改成20

print(f"全部股票串接完成！總資料筆數: {len(total_df)}")
print(f"總特徵數量: {len([col for col in total_df.columns if 'lag' in col])}")


# --- 時間序列嚴格切分 (前 80% 訓練, 後 20% 測試) ---
unique_dates = sorted(total_df['date'].unique())
split_point = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_point]
print(f"資料時間軸切分點為: {split_date}")

train_df = total_df[total_df['date'] < split_date]
test_df = total_df[total_df['date'] >= split_date]

feature_cols = [col for col in total_df.columns if col not in ['date', 'label', 'future_max_return', 'future_max_high']]
X_train, y_train = train_df[feature_cols], train_df['label']
X_test, y_test = test_df[feature_cols], test_df['label']


X_train['ticker'] = X_train['ticker'].astype('category')
X_test['ticker'] = X_test['ticker'].astype('category')
# --- 核心 LightGBM 模型配置 ---
scale_weight = 0.85 # 目前設為 1，保留後續調整不平衡樣本權重的空間
base_model = lgb.LGBMClassifier(
        objective='binary',
        metric='binary_logloss',
        scale_pos_weight=scale_weight,
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbose=-1
)

print("開始訓練多股聯合模型...")
base_model.fit(X_train, y_train, categorical_feature=['ticker'])

importances = pd.Series(base_model.feature_importances_, index=feature_cols)
print(importances.nlargest(30))

# --- 預測與統計報告 ---
y_pred = base_model.predict(X_test)
y_proba = base_model.predict_proba(X_test)[:, 1]
print("\n=== 全市場聯合模型評估報告 ===")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# --- 交易訊號過濾 ---
results = pd.DataFrame({
        'date': test_df['date'],
        'ticker': test_df['ticker'],
        'Actual_Label': y_test,
        'Confidence_Level': y_proba
}, index=test_df.index)

confidence_threshold = BUY_THRESHOLD 
high_confidence_signals = results[results['Confidence_Level'] >= confidence_threshold]
print(f"\n=== 全市場交易訊號篩選 ===")
if len(high_confidence_signals) > 0:
        actual_win_rate = high_confidence_signals['Actual_Label'].mean() * 100
        print(f"高信心度訊號的「真實跨股勝率」: {actual_win_rate:.2f}%")
        print(f"總共 {len(high_confidence_signals)}筆")

開始串接所有股票資料...
全部股票串接完成！總資料筆數: 82537
總特徵數量: 160
資料時間軸切分點為: 2025-11-04
開始訓練多股聯合模型...
ticker                     1492
max_lag_0                   176
min_lag_0                   116
min_lag_19                  104
volume_lag_3                 91
close_lag_1                  82
Trading_turnover_lag_14      79
volume_lag_19                78
volume_lag_6                 77
max_lag_19                   77
spread_lag_14                75
Trading_money_lag_3          72
Trading_money_lag_14         72
Trading_money_lag_19         72
close_lag_2                  70
volume_lag_12                70
volume_lag_16                69
Trading_turnover_lag_17      69
max_lag_1                    66
volume_lag_1                 65
spread_lag_6                 65
volume_lag_9                 65
min_lag_1                    64
min_lag_2                    64
Trading_money_lag_18         64
open_lag_19                  64
spread_lag_8                 63
volume_lag_17                63
volume_lag_18        

# 5. 回測系統輔助函式定義

封裝回測模擬中所需的底層函數，包括下單模組（考慮手續費、稅金）以及股數計算模組（支援整股交易限制）。


In [136]:
# 真實交易執行函數：動態更新帳戶可用資金與庫存持股狀態
def execute_trade (ticker, price, shares, date, action, current_capital, current_holdings) -> int | dict:
       if action == 'buy': 
              current_capital -= price * shares * (1 + ENTRY_FEE) # 扣除买入金額與買入手續費
              current_holdings[ticker] = (shares, date, price)
       else: 
              current_capital += price * shares * (1 - EXIT_FEE)  # 獲得賣出金額並扣除賣出手續費與證交稅
              del current_holdings[ticker]
       return current_capital, current_holdings

In [137]:
# 股數計算函數：支持台股整股（1000股為一張）或零股交易模式
def calculate_shares_to_buy (capital, price, isRoundLot=True) -> int:
       price *= (1 + ENTRY_FEE) 
       if isRoundLot:
              shares = capital // (price * 1000) * 1000 # 無條件捨去，計算可買進的整張張數
       else:
              shares = capital // price # 支援零股買進
       return shares

In [138]:
# 輔助資料串接函數：用以載入完整的原始 K 線資料提供給回測系統查價
def load_and_concat_data (file_list : list) -> pd.DataFrame:
       df_list = []
       for file in file_list:
              df = pd.read_csv(file)
              df_list.append(df)
       combined_df = pd.concat(df_list, ignore_index=True)
       return combined_df

In [139]:
def load_and_concat_data(file_list: list) -> pd.DataFrame:
    df_list = []
    for file in file_list:
        df = pd.read_csv(file)
        
        # 確保時間由舊到新排序，這對 shift 計算至關重要
        if 'date' in df.columns:
            df = df.sort_values(by='date').reset_index(drop=True)
        
        # 建立前一日收盤價：將 close 往下平移一格
        df['prev_close'] = df['close'].shift(1)
        
        # 處理歷史上首日無前日收盤價的極端情況（非必須，但可防 nan 造成的報錯）
        # 這裡用當天開盤價（open）或收盤價（close）暫代，避免計算漲跌停時出現 NaN
        df['prev_close'] = df['prev_close'].fillna(df['open'])
        
        df_list.append(df)
        
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

In [140]:
def get_tick_size(price: float) -> float:
    """根據台股價格區間，返回對應的升降單位(Tick Size)"""
    if price < 10.0:
        return 0.01
    elif price < 50.0:
        return 0.05
    elif price < 100.0:
        return 0.10
    elif price < 500.0:
        return 0.50
    elif price < 1000.0:
        return 1.00
    else:
        return 5.00

def get_limit_prices(prev_close: float):
    """
    根據前一日收盤價，計算精確的當日漲停價與跌停價（考慮台股 Tick 規則）
    """
    # 1. 計算理論漲跌停價
    raw_limit_up = prev_close * 1.10
    raw_limit_down = prev_close * 0.90
    
    # --- 漲停價計算 (向下尋找最接近的有效 Tick) ---
    tick_up = get_tick_size(raw_limit_up)
    # 使用整除模擬無條件捨去到 Tick 單位
    limit_up = np.floor(round(raw_limit_up / tick_up, 4)) * tick_up
    
    # --- 跌停價計算 (向上尋找最接近的有效 Tick) ---
    tick_down = get_tick_size(raw_limit_down)
    # 使用 ceil 模擬無條件進位到 Tick 單位
    limit_down = np.ceil(round(raw_limit_down / tick_down, 4)) * tick_down
    
    return round(limit_up, 2), round(limit_down, 2)

# 6. 策略 A 實戰回測模擬（單檔強勢股輪動 + 9% 動態停利）

基於模型輸出的信心度進行每日橫向掃描：

1. 檢查目前持股是否已滿足 9% 停利條件（盤中最高價觸及）或持有已滿 14 天強制到期結算。
2. 當帳戶呈現空倉時，在有高信心訊號的股票中，挑選當天最強（`proba` 最高）的個股進行全額 All-in 買進。


In [143]:
# 載入全市場完整歷史K線作為回測查價庫
stock_files = ['../data/' + stock_id + '_stock_data.csv' for stock_id in stock_ids]
whole_df = load_and_concat_data(stock_files)

backtest_df = test_df[['date', 'ticker']].copy()
backtest_df['proba'] = y_proba
backtest_df['label'] = y_pred

current_capital = INITIAL_CAPITAL
current_holdings = dict()
last_date = None

# 為了判斷當日漲跌停，回測資料必須包含「前一日收盤價」
# 假設你的 whole_df 已經有 'prev_close' 欄位（若無，可用 shift(1) 算出）

for date in backtest_df['date'].unique():
    daily_df = backtest_df[backtest_df['date'] == date]
    current_ticker = list(current_holdings.keys())[0] if current_holdings else None
    
    # --- 1. 盤中監控（賣出檢查） ---
    if current_ticker is not None:
        shares, entry_date, entry_price = current_holdings[current_ticker]
        stock_today = whole_df[(whole_df['date'] == date) & (whole_df['stock_id'] == current_ticker)].iloc[0]
        
        # 取得前日收盤價，並計算今日的精確漲跌停價
        prev_close = stock_today['prev_close']
        limit_up_price, limit_down_price = get_limit_prices(prev_close)
        
        sell_signal = False
        price_to_sell = stock_today['close']
        reason = ""
        
        # (A) 時間到期 (14天)
        if (pd.to_datetime(date) - pd.to_datetime(entry_date)).days >= 14:
            sell_signal = True
            price_to_sell = stock_today['close']
            reason = "持股滿 14 天強制平倉"
            
        # (B) 動態停損 (15%)
        elif stock_today['min'] <= entry_price * (1 + STOP_LOSS):
            # 【跌停防線】：如果今天一字線跌停鎖死，根本賣不掉！
            is_limit_down_locked = (stock_today['open'] == limit_down_price) and (stock_today['close'] == limit_down_price)
            if is_limit_down_locked:
                print(f"日期: {date} | 警告: {current_ticker} 跌停鎖死！今日無法賣出平倉，延遲至明日。")
                sell_signal = False # 取消今日賣出，等明天
            else:
                sell_signal = True
                price_to_sell = min(stock_today['open'], entry_price * (1 + STOP_LOSS))
                reason = "動態停損觸發"
                
        # (C) 動態停利 (9%)
        elif stock_today['max'] >= entry_price * (1 + TAKE_PROFIT):
            sell_signal = True
            price_to_sell = max(stock_today['open'], entry_price * (1 + TAKE_PROFIT))
            reason = "動態停利觸發"
            
        if sell_signal:
            current_capital, current_holdings = execute_trade(
                current_ticker, price_to_sell, shares, None, 'sell', current_capital, current_holdings
            )
            print(f"日期: {date} | 賣出 {current_ticker} | 價格: {price_to_sell:.2f} | 原因: {reason} | 剩餘資金: {current_capital:.2f}")
    # --- 2. 當日收盤前 5 分鐘決策並下單（買入檢查） ---
    if len(current_holdings) == 0:
        buy_df = daily_df[daily_df["label"] == 1] \
                .sort_values("proba", ascending=False)        
        if len(buy_df) > 0 and buy_df.iloc[0]['proba'] > BUY_THRESHOLD:
            ticker_to_buy = int(buy_df.iloc[0]['ticker'])
            stock_today = whole_df[(whole_df['date'] == date) & (whole_df['stock_id'] == ticker_to_buy)].iloc[0]
            
            # 取得該股今日漲停價
            prev_close = stock_today['prev_close']
            limit_up_price, _ = get_limit_prices(prev_close)
            
            # 【買入防線】：如果收盤前 5 分鐘已經牢牢鎖在漲停板，我們市價排單也買不到
            is_limit_up_locked = (stock_today['close'] == limit_up_price)
            
            if is_limit_up_locked:
                print(f"日期: {date} | 提示: {ticker_to_buy} 已鎖漲停（{limit_up_price}），今日放棄買入。")
            else:
                price_to_buy = stock_today['close'] 
                shares_to_buy = calculate_shares_to_buy(current_capital, price_to_buy, isRoundLot=True)
                
                if shares_to_buy > 0:
                    current_capital, current_holdings = execute_trade(
                        ticker_to_buy, price_to_buy, shares_to_buy, date, 'buy', current_capital, current_holdings
                    )
                    print(f"日期: {date} | 成功買入 {ticker_to_buy} | 價格: {price_to_buy:.2f} | 股數: {shares_to_buy:.0f}")

日期: 2025-11-04 | 成功買入 2408 | 價格: 131.50 | 股數: 1000
日期: 2025-11-06 | 賣出 2408 | 價格: 143.34 | 原因: 動態停利觸發 | 剩餘資金: 211170.01
日期: 2025-11-06 | 提示: 2337 已鎖漲停（34.45），今日放棄買入。
日期: 2025-11-07 | 成功買入 2408 | 價格: 147.00 | 股數: 1000
日期: 2025-11-10 | 賣出 2408 | 價格: 160.23 | 原因: 動態停利觸發 | 剩餘資金: 223656.64
日期: 2025-11-10 | 提示: 2344 已鎖漲停（63.9），今日放棄買入。
日期: 2025-11-11 | 成功買入 2408 | 價格: 164.50 | 股數: 1000
日期: 2025-11-24 | 賣出 2408 | 價格: 139.82 | 原因: 動態停損觸發 | 剩餘資金: 198301.97
日期: 2025-11-24 | 成功買入 2337 | 價格: 33.85 | 股數: 5000
日期: 2025-12-08 | 賣出 2337 | 價格: 37.25 | 原因: 持股滿 14 天強制平倉 | 剩餘資金: 214439.26
日期: 2025-12-12 | 成功買入 2344 | 價格: 74.60 | 股數: 2000
日期: 2025-12-26 | 賣出 2344 | 價格: 76.50 | 原因: 持股滿 14 天強制平倉 | 剩餘資金: 217521.88
日期: 2025-12-26 | 成功買入 2344 | 價格: 76.50 | 股數: 2000
日期: 2025-12-30 | 賣出 2344 | 價格: 83.39 | 原因: 動態停利觸發 | 剩餘資金: 230518.17
日期: 2025-12-30 | 成功買入 2344 | 價格: 83.70 | 股數: 2000
日期: 2026-01-05 | 賣出 2344 | 價格: 95.00 | 原因: 動態停利觸發 | 剩餘資金: 252242.59
日期: 2026-01-05 | 提示: 2337 已鎖漲停（47.65），今日放棄買入。
日期: 2026-01-06 | 成功